# 🧠 Agentic RAG for Healthcare

This notebook demonstrates an autonomous agentic RAG system that retrieves healthcare data, reasons using a language model, and responds interactively with a Gradio UI.

In [ ]:
# Install Required Libraries
!pip install gradio langchain faiss-cpu sentence-transformers huggingface_hub rdflib pypdf2


In [ ]:
# === Imports and Configuration ===
import os
import gradio as gr
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.text_splitter import CharacterTextSplitter
from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.agents import initialize_agent, Tool
from langchain.llms import HuggingFaceHub
from langchain.agents.agent_types import AgentType
from langchain.document_loaders import UnstructuredURLLoader

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your-huggingface-token"

llm = HuggingFaceHub(repo_id="google/flan-t5-large", model_kwargs={"temperature": 0.5, "max_length": 512})


In [ ]:
# === Document Ingestion ===
from tempfile import NamedTemporaryFile
import requests
import traceback

def load_documents(file_obj=None, url=None):
    docs = []
    try:
        if file_obj:
            ext = os.path.splitext(file_obj.name)[-1].lower()
            if ext == ".pdf":
                loader = PyPDFLoader(file_obj.name)
            else:
                loader = TextLoader(file_obj.name)
            docs.extend(loader.load())
        elif url:
            if url.lower().endswith(".pdf"):
                response = requests.get(url)
                response.raise_for_status()
                with NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
                    tmp.write(response.content)
                    tmp.flush()
                    loader = PyPDFLoader(tmp.name)
                    docs.extend(loader.load())
            else:
                loader = UnstructuredURLLoader(urls=[url])
                docs.extend(loader.load())
    except Exception as e:
        print(f"Error: {e}")
        raise
    return docs


In [ ]:
# === RAG Pipeline with Vector Store and QA Chain ===
def create_qa_chain(docs):
    splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    texts = splitter.split_documents(docs)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(texts, embeddings)
    return RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())


In [ ]:
# === Agent Setup ===
qa_chain = None
agent_executor = None

def ingest(file, url):
    global qa_chain, agent_executor
    try:
        docs = load_documents(file, url)
        if docs:
            qa_chain = create_qa_chain(docs)
            tools = [Tool(name="MedicalQATool", func=qa_chain.run, description="Useful for healthcare questions.")]
            agent_executor = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)
            return "Documents indexed successfully. Ask your questions below."
        else:
            return "No documents loaded."
    except Exception as e:
        return f"Error during ingestion: {str(e)}"


In [ ]:
# === Gradio Interface ===
def query_agent(input_text):
    if agent_executor:
        return agent_executor.run(input_text)
    return "Please load documents first."

with gr.Blocks() as demo:
    gr.Markdown("# 🏥 Agentic RAG Healthcare Chatbot")
    with gr.Row():
        file_input = gr.File(label="Upload PDF/Text")
        url_input = gr.Textbox(label="Or Enter a URL")
        load_button = gr.Button("Ingest")
    status = gr.Textbox(label="Status")
    question = gr.Textbox(label="Ask a Question")
    answer = gr.Textbox(label="Agent's Answer")

    load_button.click(ingest, inputs=[file_input, url_input], outputs=status)
    question.submit(query_agent, inputs=question, outputs=answer)

demo.launch()
